In [1]:
import json
from transformers import AutoTokenizer
import torch as t
import os
from collections import defaultdict

/fs/nexus-scratch/scheng03/miniconda3/envs/refusal/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Below Looks at the trained vector results to identify the best hyperparameter configuration

In [ ]:
model_name = 'gemma-2-2b-it'
learn_dir = f'../trained_vectors/{model_name}/'
path = learn_dir + 'ortho_bsz{bsz}_lr{lr}_seed{seed}'
batch_sizes = [6,12]
lrs = ["1e-2", "4e-2"]
seed = 42
# scalers = ["1e-5", "2e-2"]
for bsz in batch_sizes:
    for lr in lrs:
        exp_path = path.format(bsz=bsz, lr=lr, seed=seed)
        
        with open(os.path.join(exp_path, "val.json"), 'r') as f:
            val_data = json.load(f)
            best_epoch = val_data['best_epoch']
        with open(os.path.join(exp_path, f"direction_{best_epoch}_val.json"), 'r') as f:
            epoch_val_data = json.load(f)
        best_exp_loss = epoch_val_data['loss']
        best_exp_match_score = epoch_val_data['match_score']
        print(f'epoch {best_epoch}')
        print(f"bsz {bsz}, lr {lr}: loss {best_exp_loss}, match_score {best_exp_match_score}")

        v = t.load(os.path.join(exp_path, f"direction_{best_epoch}.pt"), 'cpu')


FileNotFoundError: [Errno 2] No such file or directory: '../trained_vectors/Llama-3.2-3B-Instruct/ortho_bsz6_lr1e-2_seed42/val.json'

In [6]:
model_name = 'Llama-3.2-3B-Instruct'
learn_dir = f'../trained_vectors/{model_name}/'
path = learn_dir + 'reps_bsz{bsz}_lr{lr}_ss{ss}_seed{seed}'
batch_sizes = [6,12]
lrs = ["1e-2", "4e-2"]
ss = "1e-5"
seed = 5
# scalers = ["1e-5", "2e-2"]
for bsz in batch_sizes:
    for lr in lrs:
        exp_path = path.format(bsz=bsz, lr=lr, ss=ss, seed=seed)
        best_exp_epoch = -1
        best_exp_loss = float('inf')
        # get best epoch from an exp
        for e in range(10):
            with open(os.path.join(exp_path, f"direction_{e}_val.json"), 'r') as f:
                epoch_val_data = json.load(f)
            loss = epoch_val_data['loss']
            if loss < best_exp_loss:
                best_exp_epoch = e
                best_exp_loss = loss
        # with open(os.path.join(exp_path, "val.json"), 'r') as f:
        #     val_data = json.load(f)
        #     best_epoch = val_data['best_epoch']
        with open(os.path.join(exp_path, f"direction_{best_exp_epoch}_val.json"), 'r') as f:
            epoch_val_data = json.load(f)
        best_exp_loss = epoch_val_data['loss']
        best_exp_match_score = epoch_val_data['match_score']
        print(f'epoch {best_exp_epoch}')
        print(f"bsz {bsz}, lr {lr}, ss {ss}: loss {best_exp_loss}, match_score {best_exp_match_score}")

epoch 0
bsz 6, lr 1e-2, ss 1e-5: loss 0.048127293135621585, match_score 0.7121496796607971
epoch 0
bsz 6, lr 4e-2, ss 1e-5: loss 0.1948010573614738, match_score 0.750944197177887
epoch 2
bsz 12, lr 1e-2, ss 1e-5: loss 0.028605662473182747, match_score 0.7857905030250549
epoch 0
bsz 12, lr 4e-2, ss 1e-5: loss 0.1347335841273889, match_score 0.7734770774841309


In [ ]:
reps_s5_v = t.load("trained_vectors/gemma-2-2b-it/reps_bsz6_lr1e-2_ss2e-2_seed5/direction_1.pt", map_location='cpu')
reps_s42_v = t.load("trained_vectors/gemma-2-2b-it/reps_bsz12_lr1e-2_ss2e-2_seed42/direction_9.pt", map_location='cpu')

In [ ]:
dim_v = t.load("pipeline/runs/gemma-2-2b-it/direction_layer15_pos-1.pt", map_location='cpu')